## Hyperparameter Tuning

#### DAD unsupervised

In [ ]:
import ast
import pandas as pd

df = pd.read_csv("TSB-AD/benchmark_exp/eval/HP_tuning/multi/DAD.csv")

df["HP_parsed"] = df["HP"].apply(ast.literal_eval)
df["lr"] = df["HP_parsed"].apply(lambda x: float(x.get("lr", 0)))
df["mom_score"] = df["HP_parsed"].apply(lambda x: float(x.get("mom_score", 0)))
df["win_size"] = df["HP_parsed"].apply(lambda x: int(x.get("win_size", 0)))

target_metrics = [
    "AUC-PR",
    "AUC-ROC",
    "VUS-PR",
    "VUS-ROC",
    "Standard-F1",
    "Affiliation-F",
]

config_performance = (
    df.groupby(["lr", "mom_score", "win_size"])[target_metrics]
    .mean()
    .reset_index()
)

config_performance = config_performance.sort_values(
    by="VUS-PR", ascending=False
)

print("### Average Configuration Performance Rankings ###\n")
print(config_performance.to_string(index=False))

print("\n--- Absolute Best Configuration ---")
best_config = config_performance.iloc[0]
print(
    f" Learning Rate (lr): {best_config['lr']}\n"
    f" Momentum Score (mom_score): {best_config['mom_score']}\n"
    f" Window Size (win_size): {best_config['win_size']}\n"
    f" Mean VUS-PR achieved: {best_config['VUS-PR']:.4f}\n"
    f" Mean Standard-F1 achieved: {best_config['Standard-F1']:.4f}"
)

### Merging DAD results into TSB-AD-M VUS-PR csv file

In [ ]:
import pandas as pd

df_target = pd.read_csv(
    "TSB-AD/benchmark_exp/benchmark_eval_results/multi_mergedTable_VUS-PR.csv"
)

df_source1 = pd.read_csv(
    "TSB-AD/benchmark_exp/eval/metrics/multi/DAD_Auto.csv"
)
df_source2 = pd.read_csv(
     "TSB-AD/benchmark_exp/eval/metrics/multi/DAD.csv"
)


df_target["dataset_key"] = df_target["file"].str.replace(
    ".csv", "", regex=False
)
df_source1["dataset_key"] = df_source1["file"].str.replace(
    ".csv", "", regex=False
)
df_source2["dataset_key"] = df_source2["file"].str.replace(
    ".csv", "", regex=False
)


mapping1 = df_source1.set_index("dataset_key")["VUS-PR"]
mapping2 = df_source2.set_index("dataset_key")["VUS-PR"]

data_dad_auto   = df_target["dataset_key"].map(mapping1)
data_dad        = df_target["dataset_key"].map(mapping2)

df_target.insert(loc=1, column="DAD_Auto", value=data_dad_auto)
df_target.insert(loc=2, column="DAD", value=data_dad)

df_target = df_target.drop(columns=["dataset_key"])

df_target.to_csv(
    "TSB-AD/benchmark_exp/benchmark_eval_results/multi_mergedTable_VUS-PR_updated.csv",
    index=False,
)

print("Columns added successfully in order!")

In [ ]:
import pandas as pd
df_VUS_PR = pd.read_csv('TSB-AD/benchmark_exp/benchmark_eval_results/multi_mergedTable_VUS-PR_updated.csv')
Comparaed_Solution_Pool = ['IForest', 'LOF', 'PCA', 'HBOS', 'OCSVM', 'MCD', 'KNN', 'KMeansAD', 'COPOD', 'CBLOF', 'EIF', 'RobustPCA', 'AutoEncoder', 
                    'CNN', 'LSTMAD', 'TranAD', 'AnomalyTransformer', 'OmniAnomaly', 'USAD', 'Donut', 'TimesNet', 'FITS', 'OFA', 'DAD_Auto', 'DAD']

mean_df = pd.DataFrame()
mean_df['VUS_PR_Rank'] = df_VUS_PR[Comparaed_Solution_Pool].mean().rank(ascending=False)
sorted_mean_df = mean_df.sort_values(by='VUS_PR_Rank', ascending=True)
sorted_mean_df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sorted_mean_df.index = sorted_mean_df.index.str.replace('AnomalyTransformer', 'ADTransformer')

rank_list = sorted_mean_df.index[:]

fig, ax = plt.subplots(1, 1, figsize=(8.3, 4.2))
sns.reset_orig()
rank_list = sorted_mean_df.index.to_list()

df_acc = df_VUS_PR

df_acc_plot = df_acc.rename(
    columns={
        "DAD_Auto": "DAD$_{Auto}$",
        "DAD": "DAD",
    }
)

rank_list_plot = [
    "DAD$_{Auto}$" if x == "DAD_Auto" else  x
    for x in rank_list
]

data_cols = [("AnomalyTransformer" if x == "ADTransformer" else x) for x in rank_list]

display_names = [("DAD$_{Auto}$" if x == "DAD_Auto" else ("ADTransformer" if x == "ADTransformer" else x)) for x in rank_list]

data_df = df_acc[data_cols].copy()
data_df.columns = display_names

ax = sns.boxplot(
    data=data_df,
    showfliers=False,
    meanprops=dict(color='k', linestyle='--'),
    showmeans=True,
    meanline=True
)


extracted_colors = [patch.get_facecolor() for patch in ax.patches]
color_mapping = dict(zip(rank_list, extracted_colors))

hex_color_mapping = {
    method: f"#{int(c[0]*255):02x}{int(c[1]*255):02x}{int(c[2]*255):02x}"
    for method, c in color_mapping.items()
}

for tick in ax.get_xticklabels():
    if tick.get_text() in ["DAD$_{Auto}$", "DAD", "DADS"]:
        tick.set_fontweight("bold")

plt.xticks(ticks=range(len(rank_list_plot)), labels=rank_list_plot, rotation=90, fontsize=12)
plt.ylabel('VUS-PR', fontsize=12)
plt.tight_layout()
plt.savefig('TSB-AD/benchmark_exp/eval/figures/multi_VUS_PR_boxplot.pdf', dpi=1200, bbox_inches='tight')
plt.show()


### Critical Difference Functions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import os
import operator
import math
from scipy.stats import friedmanchisquare
from scikit_posthocs import posthoc_nemenyi_friedman
import networkx
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg
import seaborn as sns

def Friedman_Nemenyi(alpha=0.05, df_perf=None):
    df_counts = pd.DataFrame({'count': df_perf.groupby(
        ['classifier_name']).size()}).reset_index()
    # Record the maximum number of datasets
    max_nb_datasets = df_counts['count'].max()
    # Create a list of classifiers
    classifiers = list(df_counts.loc[df_counts['count'] == max_nb_datasets]
                       ['classifier_name'])

    # print('classifiers: ', classifiers)

    '''
    Expected input format for friedmanchisquare is:
                Dataset1        Dataset2        Dataset3        Dataset4        Dataset5
    classifer1
    classifer2
    classifer3 
    '''

    # Compute friedman p-value
    friedman_p_value = friedmanchisquare(*(
        np.array(df_perf.loc[df_perf['classifier_name'] == c]['accuracy'])
        for c in classifiers))[1]

    # Decide whether to reject the null hypothesis
    # If p-value >= alpha: we cannot reject the null hypothesis. No statistical difference.
    if friedman_p_value >= alpha:
        print('No statistical difference...')
        return None,None,None
    # Friedman test OK
    # Prepare input for Nemenyi test
    data = []
    for c in classifiers:
        data.append(df_perf.loc[df_perf['classifier_name'] == c]['accuracy'])
    data = np.array(data, dtype=np.float64)
    # Conduct the Nemenyi post-hoc test
    # print(classifiers)
    # Order is classifiers' order
    nemenyi = posthoc_nemenyi_friedman(data.T)

    # print(nemenyi)
    
    # Original code: p_values.append((classifier_1, classifier_2, p_value, False)), True: represents there exists statistical difference
    p_values = []

    # Comparing p-values with the alpha value
    for nemenyi_indx in nemenyi.index:
        for nemenyi_columns in nemenyi.columns:
            if nemenyi_indx < nemenyi_columns:
                if nemenyi.loc[nemenyi_indx, nemenyi_columns] < alpha:
                    p_values.append((classifiers[nemenyi_indx], classifiers[nemenyi_columns], nemenyi.loc[nemenyi_indx, nemenyi_columns], True))
                else:
                    p_values.append((classifiers[nemenyi_indx], classifiers[nemenyi_columns], nemenyi.loc[nemenyi_indx, nemenyi_columns], False))
            else: continue

    # Nemenyi test OK

    m = len(classifiers)

    # Sort by classifier name then by dataset name
    sorted_df_perf = df_perf.loc[df_perf['classifier_name'].isin(classifiers)]. \
        sort_values(['classifier_name', 'dataset_name'])

    rank_data = np.array(sorted_df_perf['accuracy']).reshape(m, max_nb_datasets)

    df_ranks = pd.DataFrame(data=rank_data, index=np.sort(classifiers), columns=np.unique(sorted_df_perf['dataset_name']))

    dfff = df_ranks.rank(ascending=False)
    # compute average rank
    average_ranks = df_ranks.rank(ascending=False).mean(axis=1).sort_values(ascending=False)
    
    return p_values, average_ranks, max_nb_datasets

# def graph_ranks(avranks, names, p_values, cd=None, cdmethod=None, lowv=None, highv=None,
#                 width=200, textspace=1, reverse=False, filename=None, **kwargs):
def graph_ranks(avranks, names, p_values, cd=None, cdmethod=None, lowv=None, highv=None,
                width=200, textspace=1, reverse=False, filename=None, ax=None, font_size=12, **kwargs):    
    width = width
    textspace = float(textspace)
    '''l is an array of array 
        [[......]
         [......]
         [......]]; 
    n is an integer'''
    # n th column
    def nth(l, n):
        n = lloc(l, n)
        # Return n th column
        return [a[n] for a in l]
    
    '''l is an array of array 
        [[......]
         [......]
         [......]]; 
    n is an integer'''
    # return an integer, count from front or from back.
    def lloc(l, n):
        if n < 0:
            return len(l[0]) + n
        else:
            return n
    # lr is an array of integers
    # Maximum range start from all zeros. Returns an iterable element of tuple.
    def mxrange(lr):
        # If nothing in the array
        if not len(lr):
            yield ()
        else:
            index = lr[0]
            # Check whether index is an integer.
            if isinstance(index, int):
                index = [index]
            # *index: index must be an iterable []
            for a in range(*index):
                for b in mxrange(lr[1:]):
                    # Form a tuple, and generate an iterable value
                    yield tuple([a] + list(b))

    def print_figure(fig, *args, **kwargs):
        canvas = FigureCanvasAgg(fig)
        canvas.print_figure(*args, **kwargs)

    sums = avranks

    nnames = names
    ssums = sums
    # lowv: low value
    if lowv is None:
        '''int(math.floor(min(ssums))): select the minimum value in ssums and take floor.
           Then compare with 1 to see which one is the minimum.'''
        lowv = min(1, int(math.floor(min(ssums))))
    # highv: high value
    if highv is None:
        highv = max(len(avranks), int(math.ceil(max(ssums))))

    cline = 0.9
    # how many algorithms
    k = len(sums)

    lines = None

    linesblank = 0
    scalewidth = width - 2 * textspace
    
    # Position of rank
    def rankpos(rank):
        if not reverse:
            a = rank - lowv
        else:
            a = highv - rank
        # Set up the format
        return textspace + scalewidth / (highv - lowv) * a

    distanceh = 0.25

    cline += distanceh

    # set up the formats
    minnotsignificant = max(2 * 0.085, linesblank)
    height = cline + ((k + 1) / 2) * 0.085 + minnotsignificant + 1.9

    if ax is None:
        fig = plt.figure(figsize=(width, height))
        fig.set_facecolor('white')
        ax = fig.add_axes([0, 0, 1, 1])
    else:
        fig = ax.get_figure()

    # matplotlib figure format setup
    # fig = plt.figure(figsize=(width, height))
    # fig.set_facecolor('white')
    # ax = fig.add_axes([0, 0, 1, 1])
    
    ax.set_axis_off()

    hf = 1. / height
    wf = 1. / width

    def hfl(l):
        return [a * hf for a in l]

    def wfl(l):
        return [a * wf for a in l]

    
    ax.plot([0, 1], [0, 1], c="w")
    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0)

    # Line plots
    def line(l, color='k', **kwargs):
        ax.plot(wfl(nth(l, 0)), hfl(nth(l, 1)), color=color, **kwargs)

    # Add text to the plot
    def text(x, y, s, bold=False, *args, **kwargs):
        ax.text(wf * x, hf * y, s, fontweight='bold' if bold else 'normal', *args, **kwargs)

    line([(textspace, cline), (width - textspace, cline)], linewidth=0.7)

    bigtick = 0.1
    smalltick = 0.05
    linewidth = 1.0
    linewidth_sign = 2.0

    tick = None

    # [lowv, highv], step size is 0.5
    for a in list(np.arange(lowv, highv, 0.5)) + [highv]:
        tick = smalltick
        # If a is an integer
        if a == int(a):
            tick = bigtick
        # Plot a line
        line([(rankpos(a), cline - tick / 2),
              (rankpos(a), cline)],
             linewidth=0.7)

    # Add text to the plot, only for integer value
    for a in range(lowv, highv + 1):
        text(rankpos(a), cline - tick / 2 - 0.05, str(a),
             ha="center", va="bottom", size=font_size)

    k = len(ssums)

    def filter_names(name):
        return name

    space_between_names = 0.24

    # Format for the first half of algorithms
    for i in range(math.ceil(k / 2)):
        chei = cline + minnotsignificant + i * space_between_names
        line([(rankpos(ssums[i]), cline),
              (rankpos(ssums[i]), chei),
              (textspace - 0.1, chei)],
             linewidth=linewidth)

        color = 'k'
        text(textspace - 0.2, chei, filter_names(nnames[i]), color=color, ha="right", va="center", size=font_size, bold=True if nnames[i] in ["DAD$_{Auto}$", "DAD", "DADS"] else False)
        # text(textspace - 0.2, chei, filter_names(name_mapping[nnames[i]] if nnames[i] in name_mapping.keys() else nnames[i]), color=color, ha="right", va="center", size=16)


    # Format for the second half of algorithms
    for i in range(math.ceil(k / 2), k):
        chei = cline + minnotsignificant + (k - i - 1) * space_between_names
        line([(rankpos(ssums[i]), cline),
              (rankpos(ssums[i]), chei),
              (textspace + scalewidth + 0.1, chei)],
             linewidth=linewidth)

        color = 'k'
        text(textspace + scalewidth + 0.2, chei, filter_names(nnames[i]), color=color, ha="left", va="center", size=font_size, bold=True if nnames[i] in ["DAD$_{Auto}$", "DAD", "DADS"] else False)
        # text(textspace + scalewidth + 0.2, chei, filter_names(name_mapping[nnames[i]] if nnames[i] in name_mapping.keys() else nnames[i]), color=color, ha="left", va="center", size=16)
        

    # no-significance lines
    def draw_lines(lines, side=0.05, height=0.1):
        start = cline + 0.2

        for l, r in lines:
            line([(rankpos(ssums[l]) - side, start),
                  (rankpos(ssums[r]) + side, start)],
                 linewidth=linewidth_sign)
            start += height
            
    start = cline + 0.2
    side = -0.02
    height = 0.1


    #Generate cliques and plot a line to connect elements in cliques    
    cliques = form_cliques(p_values, nnames)
    i = 1
    achieved_half = False
    # Plot a line to connect elements in cliques
    for clq in cliques:
        if len(clq) == 1:
            continue
        min_idx = np.array(clq).min()
        max_idx = np.array(clq).max()
        if min_idx >= len(nnames) / 2 and achieved_half == False:
            start = cline + 0.25
            achieved_half = True
        # Test
        # print("ssums[min_idx]: {}; ssums[max_idx]: {}".format(ssums[min_idx], ssums[max_idx]))
        line([(rankpos(ssums[min_idx]) - side, start),
              (rankpos(ssums[max_idx]) + side, start)],
             linewidth=linewidth_sign)
        start += height

def form_cliques(p_values, nnames):
    m = len(nnames)
    g_data = np.zeros((m, m), dtype=np.int64)
    for p in p_values:
        if p[3] == False:
            i = np.where(nnames == p[0])[0][0]
            j = np.where(nnames == p[1])[0][0]
            min_i = min(i, j)
            max_j = max(i, j)
            g_data[min_i, max_j] = 1
    g = networkx.Graph(g_data)

    return networkx.find_cliques(g)

#### CD + Box-plot figure

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

fig, ax = plt.subplots(
    2, 1, 
    figsize=(9, 6.75), 
    gridspec_kw={'height_ratios': [0.6, 0.4], 'hspace': 0.02}
) 
sns.reset_orig()

df_acc = df_VUS_PR

eval_list = []
for index, row in df_acc.iterrows():
    for method in Comparaed_Solution_Pool:
        eval_list.append([method, row['file'], row[method]])
eval_df = pd.DataFrame(eval_list, columns=['classifier_name', 'dataset_name', 'accuracy'])

eval_df['classifier_name'] = eval_df['classifier_name'].replace({'DAD_Auto': 'DAD$_{Auto}$', "AnomalyTransformer": "ADTransformer"})

p_values, average_ranks, _ = Friedman_Nemenyi(df_perf=eval_df, alpha=0.05)
ranking = average_ranks.keys().to_list()[::-1]
ranking = ['ADTransformer' if x == 'AnomalyTransformer' else x for x in ranking]

graph_ranks(average_ranks.values, average_ranks.keys(), p_values, cd=None, 
            reverse=True, width=30, textspace=2.0, ax=ax[0])


rank_list = sorted_mean_df.index.to_list()

df_acc_plot = df_acc.rename(
    columns={
        "DAD_Auto": "DAD$_{Auto}$",
        "AnomalyTransformer": "ADTransformer",
        "DAD": "DAD",
    }
)

rank_list_plot = [
    "DAD$_{Auto}$" if x == "DAD_Auto" else x
    for x in rank_list
]

sns.boxplot(
    data=df_acc_plot[rank_list_plot],
    showfliers=False,
    meanprops=dict(color='k', linestyle='--'),
    showmeans=True,
    meanline=True,
    ax=ax[1]  
)

ax[1].set_xticks(ticks=range(len(rank_list_plot)))
ax[1].set_xticklabels(rank_list_plot, rotation=90, fontsize=12)
ax[1].set_ylabel('VUS-PR', fontsize=12)

for tick in ax[1].get_xticklabels():
    if tick.get_text() in ["DAD$_{Auto}$", "DAD", "DADS"]:
        tick.set_fontweight("bold")

plt.subplots_adjust(top=0.98, bottom=0.25, left=0.06, right=0.94)

output_path = 'TSB-AD/benchmark_exp/eval/figures/streaming_benchmark_main.pdf'
plt.savefig(output_path, dpi=1200, bbox_inches='tight')
plt.show()

#### Outperformings Table

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon

def outperformance_table(df_acc, methods, alpha_strong=0.05, alpha_weak=0.10):
    """
    Build a pairwise outperformance table.

    Cell (row=A, col=B):
      '++'  row A significantly outperforms col B at alpha=0.05
      '+'   row A significantly outperforms col B at alpha=0.10
      '--'  row A significantly underperforms col B at alpha=0.05
      '-'   row A significantly underperforms col B at alpha=0.10
      '~'   no significant difference
      '-'   diagonal

    Uses a one-sided Wilcoxon signed-rank test across datasets.
    """
    n = len(methods)
    # score matrix: rows=datasets, cols=methods
    scores = df_acc[methods].values  # shape (n_datasets, n_methods)

    symbols = pd.DataFrame('~', index=methods, columns=methods)

    for i, m_a in enumerate(methods):
        for j, m_b in enumerate(methods):
            if i == j:
                symbols.loc[m_a, m_b] = '$-$'
                continue
            diff = scores[:, i] - scores[:, j]
            if np.all(diff == 0):
                symbols.loc[m_a, m_b] = '~'
                continue
            # explicit one-sided tests — avoids p/2 approximation with ties
            _, p_greater = wilcoxon(diff, alternative='greater')
            _, p_less    = wilcoxon(diff, alternative='less')

            if p_greater <= alpha_strong:
                symbols.loc[m_a, m_b] = '$++$'
            elif p_greater <= alpha_weak:
                symbols.loc[m_a, m_b] = '$+$'
            elif p_less <= alpha_strong:
                symbols.loc[m_a, m_b] = '$--$'
            elif p_less <= alpha_weak:
                symbols.loc[m_a, m_b] = '$-$'
            else:
                symbols.loc[m_a, m_b] = '~'

    return symbols



table = outperformance_table(df_acc, Comparaed_Solution_Pool)

table.index = table.index.str.replace('DAD_Auto', 'DAD$_{Auto}$')
table.columns = table.columns.str.replace('DAD_Auto', 'DAD$_{Auto}$')
table.index = table.index.str.replace('AnomalyTransformer', 'ADTransformer')
table.columns = table.columns.str.replace('AnomalyTransformer', 'ADTransformer')

table = table.loc[ranking, ranking]

table['Outperformances'] = (table == '$++$').sum(axis=1)
table = table.sort_values('Outperformances', ascending=False).drop(columns='Outperformances')
table = table[table.index] 
table['Outperformances'] = (table == '$++$').sum(axis=1)
table.columns = [f'\\rotatebox{{90}}{{{col}}}' for col in table.columns]
print(table.to_latex())

#### Detailed method performances across datasets

In [ ]:
import pandas as pd

df_VUS_PR = pd.read_csv('TSB-AD/benchmark_exp/benchmark_eval_results/multi_mergedTable_VUS-PR_updated.csv')

Comparaed_Solution_Pool = [
    'IForest', 'LOF', 'PCA', 'HBOS', 'OCSVM', 'MCD', 'KNN', 'KMeansAD', 'COPOD', 
    'CBLOF', 'EIF', 'RobustPCA', 'AutoEncoder', 'CNN', 'LSTMAD', 'TranAD', 
    'AnomalyTransformer', 'OmniAnomaly', 'USAD', 'Donut', 'TimesNet', 'FITS', 'OFA', 
    'DAD_Auto', 'DAD'
]

df_VUS_PR['dataset'] = df_VUS_PR['file'].apply(lambda x: x.split('_id_')[0].split('_', 1)[1])

mean_df = df_VUS_PR.groupby('dataset')[Comparaed_Solution_Pool].mean().reset_index()
mean_df_int = mean_df.copy()
global_avg = mean_df[Comparaed_Solution_Pool].mean()
global_avg_row = pd.DataFrame([['Average'] + global_avg.tolist()], columns=['dataset'] + Comparaed_Solution_Pool)
mean_df = pd.concat([mean_df, global_avg_row], ignore_index=True)

decimal_places = 2 
mean_df[Comparaed_Solution_Pool] = mean_df[Comparaed_Solution_Pool].round(decimal_places)
mean_df = mean_df.set_index('dataset').T.reset_index().rename(columns={'index': 'classifier_name'})
mean_df = mean_df.sort_values(by='Average', ascending=False).reset_index(drop=True)

mean_df['classifier_name'] = mean_df['classifier_name'].replace('DAD_Auto', 'DAD$_{Auto}$')
mean_df['classifier_name'] = mean_df['classifier_name'].replace('AnomalyTransformer', 'ADTransformer')
mean_df = mean_df.rename(columns={'classifier_name': 'Method'})
mean_df.columns = ['Method'] + [f'\\rotatebox{{90}}{{{col}}}' for col in mean_df.columns[1:]]
latex_table = mean_df.to_latex(index=False, float_format="%.2f")
print(latex_table)

#### DAD variants performance comparison 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

considered_methods = ["DAD", "DAD_Auto"]
mean_df_pre = mean_df_int[['dataset'] + considered_methods]

mean_df_pre = mean_df_pre.rename(columns={'classifier_name': 'Method'})
PerfGAP = mean_df_pre['DAD_Auto'] - mean_df_pre['DAD']

colors = ['#1f77b4' if val >= 0 else '#ff7f0e' for val in PerfGAP]

fig, ax = plt.subplots(figsize=(6.5, 2.7))
ax.set_axisbelow(True)
plt.grid(axis='y', linestyle='--', alpha=0.99)
sns.barplot(x=mean_df_pre['dataset'], y=PerfGAP, palette=colors, ax=ax)
plt.xticks(ticks=plt.xticks()[0], labels=mean_df_pre['dataset'].astype(str).str.replace("OPPORTUNITY", "OPPOR.", regex=False), rotation=90)
plt.ylabel('VUS-PR Gain')
plt.xlabel('')
sns.despine(top=True, right=True)
plt.tight_layout()
plt.savefig('TSB-AD/benchmark_exp/eval/figures/performance_gap_DAD_Auto_vs_DAD.pdf', dpi=1200, bbox_inches='tight')
plt.show()

#### Dimensionality Figure

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_VUS_PR = pd.read_csv('TSB-AD/benchmark_exp/benchmark_eval_results/multi_mergedTable_VUS-PR_updated.csv')

Comparaed_Solution_Pool = [
    'IForest', 'LOF', 'PCA', 'HBOS', 'OCSVM', 'MCD', 'KNN', 'KMeansAD', 'COPOD', 
    'CBLOF', 'EIF', 'RobustPCA', 'AutoEncoder', 'CNN', 'LSTMAD', 'TranAD', 
    'AnomalyTransformer', 'OmniAnomaly', 'USAD', 'Donut', 'TimesNet', 'FITS', 'OFA', 
    'DAD_Auto', 'DAD'
]

selected_files = [
    '123_TAO_id_8_Environment_tr_500_1st_62.csv', 
    '052_GHL_id_21_Sensor_tr_50000_1st_98001.csv', 
    '076_SMD_id_20_Facility_tr_5925_1st_17580.csv', 
    '016_MSL_id_15_Sensor_tr_500_1st_780.csv', 
    '133_OPPORTUNITY_id_5_HumanActivity_tr_1745_1st_6500.csv'
]

case_df = df_VUS_PR[df_VUS_PR['file'].isin(selected_files)].set_index('file').reindex(selected_files)

user_specified_methods = ['DAD_Auto', 'DAD', 'CNN', 'OmniAnomaly'] 
if user_specified_methods and len(user_specified_methods) > 0:
    plot_methods = user_specified_methods
    print(f"Using user-defined methods for plotting: {plot_methods}")
else:
    top_methods = case_df[Comparaed_Solution_Pool].mean(axis=0).sort_values(ascending=False).head(5)
    plot_methods = top_methods.index.tolist()
    print(f"No custom list provided. Automatically selected top 5 methods: {plot_methods}")

scores = [case_df[method].round(2).tolist() for method in plot_methods]

dataset_labels = [
    'TAO\n($d=3$)', 
    'GHL\n($d=19$)', 
    'SMD\n($d=38$)', 
    'MSL\n($d=55$)', 
    'OPPORTUNITY\n($d=248$)'
]

method_labels = [
    m.replace('DAD_Auto', 'DAD$_{Auto}$')
     .replace('KNN', '$k$-NN')
     .replace('IForest', 'I-Forest') 
    for m in plot_methods
]

default_colors = ["#435d7c", "#f18f38", "#bc0c6a", "#5aa4c2"]

if len(plot_methods) <= len(default_colors):
    colors = default_colors[:len(plot_methods)]
else:
    cmap = plt.get_cmap('tab20')
    colors = [cmap(i) for i in np.linspace(0, 1, len(plot_methods))]

group_coverage = 0.77  

x = np.arange(len(dataset_labels))
width = group_coverage / len(plot_methods) 
offsets = np.linspace(-((len(plot_methods)-1)/2), (len(plot_methods)-1)/2, len(plot_methods)) * width

fig, ax = plt.subplots(figsize=(11, 3.25))

for i, (method, color) in enumerate(zip(method_labels, colors)):
    values = scores[i]
    ax.bar(x + offsets[i], values, width, label=method, color=color)
    
    
    for xi, val in zip(x + offsets[i], values):
        ax.text(xi, val + 0.03, f"{val:.2f}", ha='center', va='bottom', fontsize=16, rotation=90)

ax.set_xticks(x)
ax.set_xticklabels(dataset_labels, fontsize=16)
ax.set_ylim(0.0, 1.12) 

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)      
ax.spines['bottom'].set_visible(True)    
ax.spines['bottom'].set_linewidth(2)

ax.yaxis.set_visible(False)          

plt.legend(
    title='',
    loc='lower center',
    bbox_to_anchor=(0.5, 1.3),         
    fontsize=18,
    frameon=False,
    ncol=len(plot_methods)
)

plt.tight_layout()
fig.savefig('TSB-AD/benchmark_exp/eval/figures/highdim_analysis_streaming.pdf', dpi=1200, bbox_inches='tight', pad_inches=0.0)
plt.show()

#### Comprehensive metrics summary

In [ ]:
import numpy as np
import pandas as pd

df = pd.DataFrame(
    {
        "Method": [
            "CNN",
            "OmniAnomaly",
            "PCA",
            "LSTMAD",
            "USAD",
            "AutoEncoder",
            "KMeansAD",
            "CBLOF",
            "MCD",
            "OCSVM",
            "Donut",
            "RobustPCA",
            "FITS",
            "OFA",
            "EIF",
            "COPOD",
            "IForest",
            "HBOS",
            "TimesNet",
            "KNN",
            "TranAD",
            "LOF",
            "AnomalyTransformer",
        ],
        "AUC-PR": [
            0.32,
            0.27,
            0.31,
            0.31,
            0.26,
            0.30,
            0.25,
            0.28,
            0.27,
            0.23,
            0.20,
            0.24,
            0.15,
            0.15,
            0.19,
            0.20,
            0.19,
            0.16,
            0.13,
            0.14,
            0.14,
            0.10,
            0.07,
        ],
        "AUC-ROC": [
            0.73,
            0.65,
            0.70,
            0.70,
            0.64,
            0.67,
            0.69,
            0.67,
            0.65,
            0.61,
            0.64,
            0.58,
            0.58,
            0.55,
            0.67,
            0.65,
            0.66,
            0.63,
            0.56,
            0.51,
            0.59,
            0.53,
            0.52,
        ],
        "VUS-PR": [
            0.31,
            0.31,
            0.31,
            0.31,
            0.30,
            0.30,
            0.29,
            0.27,
            0.27,
            0.26,
            0.26,
            0.24,
            0.21,
            0.21,
            0.21,
            0.20,
            0.20,
            0.19,
            0.19,
            0.18,
            0.18,
            0.14,
            0.12,
        ],
        "VUS-ROC": [
            0.76,
            0.69,
            0.74,
            0.74,
            0.68,
            0.69,
            0.73,
            0.70,
            0.69,
            0.67,
            0.71,
            0.61,
            0.66,
            0.63,
            0.71,
            0.69,
            0.69,
            0.67,
            0.64,
            0.59,
            0.65,
            0.60,
            0.57,
        ],
        "Standard-F1": [
            0.37,
            0.32,
            0.37,
            0.36,
            0.31,
            0.34,
            0.31,
            0.32,
            0.33,
            0.28,
            0.28,
            0.29,
            0.22,
            0.21,
            0.26,
            0.27,
            0.26,
            0.24,
            0.20,
            0.19,
            0.21,
            0.15,
            0.12,
        ],
        "PA-F1": [
            0.78,
            0.55,
            0.79,
            0.79,
            0.53,
            0.60,
            0.68,
            0.65,
            0.46,
            0.48,
            0.52,
            0.60,
            0.72,
            0.72,
            0.74,
            0.72,
            0.68,
            0.67,
            0.68,
            0.69,
            0.68,
            0.57,
            0.53,
        ],
        "Event-based-F1": [
            0.65,
            0.41,
            0.59,
            0.64,
            0.40,
            0.44,
            0.49,
            0.45,
            0.33,
            0.41,
            0.36,
            0.42,
            0.32,
            0.41,
            0.44,
            0.41,
            0.41,
            0.40,
            0.32,
            0.45,
            0.40,
            0.32,
            0.33,
        ],
        "R-based-F1": [
            0.37,
            0.37,
            0.29,
            0.38,
            0.37,
            0.28,
            0.33,
            0.31,
            0.20,
            0.30,
            0.21,
            0.33,
            0.16,
            0.17,
            0.26,
            0.24,
            0.24,
            0.24,
            0.17,
            0.21,
            0.21,
            0.14,
            0.14,
        ],
        "Affiliation-F": [
            0.87,
            0.81,
            0.85,
            0.87,
            0.80,
            0.80,
            0.82,
            0.81,
            0.76,
            0.80,
            0.81,
            0.81,
            0.81,
            0.83,
            0.81,
            0.80,
            0.80,
            0.80,
            0.82,
            0.79,
            0.79,
            0.76,
            0.74,
        ],
    }
)

new_files = {
    "DAD_Auto": "TSB-AD/benchmark_exp/eval/metrics/multi/DAD_Auto.csv",
    "DAD": "TSB-AD/benchmark_exp/eval/metrics/multi/DAD.csv",
}

metrics_to_average = [col for col in df.columns if col != "Method"]

new_rows = []

for method_name, path in new_files.items():
    try:
        temp_df = pd.read_csv(path)
        means = temp_df[metrics_to_average].mean()
        row_data = {"Method": method_name}
        row_data.update(means.to_dict())
        new_rows.append(row_data)
        print(f"Successfully processed {method_name} (N={len(temp_df)})")
    except Exception as e:
        print(f"Error processing {method_name} at {path}: {e}")

if new_rows:
    df_new_rows = pd.DataFrame(new_rows)
    df = pd.concat([df, df_new_rows], ignore_index=True)


for metric in metrics_to_average:
    best_value = df[metric].max()
    second_best_value = df[metric].nlargest(2).iloc[-1]
    df[metric] = df[metric].apply(lambda x: f"\\textbf{{{x:.2f}}}" if x == best_value else (f"\\underline{{{x:.2f}}}" if x == second_best_value else f"{x:.2f}"))

desired_order = ['DAD_Auto', 'DAD', 'DADS']
df['Method'] = pd.Categorical(df['Method'], categories=desired_order + [m for m in df['Method'] if m not in desired_order], ordered=True)
df = df.sort_values('Method').reset_index(drop=True)

df['Method'] = df['Method'].replace({
    'DAD_Auto': 'DAD$_{Auto}$',
    'DAD': 'DAD',
    'DADS': 'DADS'
})

df = df.rename(columns={'Affiliation-F': 'Affiliation-F1'})
latex_table = df.to_latex(index=False, float_format="%.2f")
print("\n--- Generated LaTeX Table ---")
print(latex_table)

#### FLOPs and Parameters Figure

In [ ]:
import torch
from anomalydetection.TPAMI.Final.TSB_AD.TSB_AD.models.CNN import CNN
from TSB_AD.models.OmniAnomaly import OmniAnomaly
import numpy as np
import matplotlib.pyplot as plt
from ptflops import get_model_complexity_info
from torch.utils.flop_counter import FlopCounterMode

FEATURE_SIZES = [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048] 
CNN_WIN       = 50
OMNI_WIN      = 100
CNN_CHAN      = [32, 32, 40]
DAD_P         = 1                    

FIG_W, FIG_H  = 7.2, 2.9             
DPI           = 1200
FONT_SIZE     = 12
LABEL_SIZE    = 14
TICK_SIZE     = 11
LEGEND_SIZE   = 12
LINE_W        = 1.7
MARKER_SIZE   = 4.5
GRID_ALPHA    = 0.35
SAVE_PATH     = "TSB-AD/benchmark_exp/eval/figures/complexity_vs_d.pdf"

ORDER  = ["DAD$_{Auto}$ (Act)", "DAD$_{Auto}$ (Psv)", "CNN", "OmniAnomaly"]
COLORS = {"DAD$_{Auto}$ (Act)": "#f18f38", "DAD$_{Auto}$ (Psv)": "#233F68", "CNN": "#bc0c6a", "OmniAnomaly": "#5aa4c2"}
MARKERS = {"DAD$_{Auto}$ (Act)": "o", "DAD$_{Auto}$ (Psv)": "+", "CNN": "s", "OmniAnomaly": "^"}
# ========================================================================

plt.rcParams.update({
    "font.size": FONT_SIZE,
    "axes.labelsize": LABEL_SIZE,
    "xtick.labelsize": TICK_SIZE,
    "ytick.labelsize": TICK_SIZE,
    "legend.fontsize": LEGEND_SIZE,
    "axes.linewidth": 0.8,
})

def dad_flops_params(d, p=1, device="cpu"):
    X = torch.randn(p, d, device=device)
    R = torch.eye(d, device=device)
    with FlopCounterMode(display=False) as fc:
        Xhat  = X @ R.t()
        C     = (Xhat.t() @ Xhat) / p
        C_off = C - torch.diag(torch.diag(C))
        upd   = C_off @ R
        R_new = R - (0.1 / (d - 1)) * upd
        score = torch.linalg.norm(C)
    return fc.get_total_flops(), d*d


def dad_flops_params_passive(d, p=1, device="cpu"):
    X = torch.randn(p, d, device=device)
    R = torch.eye(d, device=device)
    with FlopCounterMode(display=False) as fc:
        Xhat  = X @ R.t()
        score = torch.linalg.norm(Xhat)
    return fc.get_total_flops(), d*d

def neural_flops_params(build_model, win, d):
    model = build_model(d).cpu().eval()
    macs, params = get_model_complexity_info(
        model, (win, d), as_strings=False,
        print_per_layer_stat=False, verbose=False,
    )
    return 2 * int(macs), int(params + win*d)     


build_cnn  = lambda d: CNN(window_size=CNN_WIN, num_channel=CNN_CHAN,
                           feats=d, lr=0.0008, batch_size=128).model
build_omni = lambda d: OmniAnomaly(win_size=OMNI_WIN, feats=d, lr=0.002).model


# ---------- Sweep ----------
flops = {m: [] for m in ORDER}
params = {m: [] for m in ORDER}

for d in FEATURE_SIZES:
    f_dad, p_dad = dad_flops_params(d)
    f_passive, p_passive = dad_flops_params_passive(d, p=DAD_P)
    f_cnn, p_cnn = neural_flops_params(build_cnn, CNN_WIN, d)
    f_omn, p_omn = neural_flops_params(build_omni, OMNI_WIN, d)

    flops["DAD$_{Auto}$ (Act)"].append(f_dad);  params["DAD$_{Auto}$ (Act)"].append(p_dad)
    flops["DAD$_{Auto}$ (Psv)"].append(f_passive);  params["DAD$_{Auto}$ (Psv)"].append(p_passive)
    flops["CNN"].append(f_cnn);         params["CNN"].append(p_cnn)
    flops["OmniAnomaly"].append(f_omn); params["OmniAnomaly"].append(p_omn)

# ---------- Print table ----------
print(f"{'d':>5} | {'DAD$_{Auto}$ (Act) FLOPs':>12} {'DAD$_{Auto}$ (Psv) FLOPs':>12} {'CNN FLOPs':>12} {'Omni FLOPs':>12} "
      f"| {'DAD$_{Auto}$ (Act) par':>9} {'DAD$_{Auto}$ (Psv) par':>9} {'CNN par':>9} {'Omni par':>9}")
for i, d in enumerate(FEATURE_SIZES):
    print(f"{d:>5} | {flops['DAD$_{Auto}$ (Act)'][i]:>12,d} {flops['DAD$_{Auto}$ (Psv)'][i]:>12,d} {flops['CNN'][i]:>12,d} "
          f"{flops['OmniAnomaly'][i]:>12,d} | {params['DAD$_{Auto}$ (Act)'][i]:>9,d} {params['DAD$_{Auto}$ (Psv)'][i]:>9,d} "
          f"{params['CNN'][i]:>9,d} {params['OmniAnomaly'][i]:>9,d}")


# ---------- Plot ----------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(FIG_W, FIG_H))
x = np.array(FEATURE_SIZES)

for m in ORDER:
    ax1.plot(x, flops[m], marker=MARKERS[m], color=COLORS[m], linestyle='--',
             lw=LINE_W, ms=MARKER_SIZE, label=m)
    ax2.plot(x, params[m], marker=MARKERS[m], color=COLORS[m], linestyle='--',
             lw=LINE_W, ms=MARKER_SIZE, label=m)

for ax, ylab in ((ax1, "FLOPs per step"), (ax2, "Parameters")):
    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.set_xticks(x)
    ax.set_xticklabels(x, rotation=40)
    ax.set_xlabel(r"Feature dimension $d$")
    ax.set_ylabel(ylab)
    ax.grid(True, which="major", alpha=GRID_ALPHA)
    ax.grid(True, which="minor", alpha=GRID_ALPHA * 0.4)

handles, labels = ax1.get_legend_handles_labels()
df_VUS_PRs = {
    "DAD$_{Auto}$ (Act)": 0.38,
    "DAD$_{Auto}$ (Psv)": 0.38,
    "CNN": 0.31,
    "OmniAnomaly": 0.31
}
F1s = {
    "DAD$_{Auto}$ (Act)": 0.43,
    "DAD$_{Auto}$ (Psv)": 0.44,
    "CNN": 0.37,
    "OmniAnomaly": 0.32
}

labels = [f"{lab}\nVUS-PR: {df_VUS_PRs.get(lab, 0):.2f}\nF1: {F1s.get(lab, 0):.2f}" for lab in labels]
fig.legend(handles, labels, ncol=4, loc="upper center",
           bbox_to_anchor=(0.5, 1.25), columnspacing=0.55, frameon=False)
fig.tight_layout()
fig.savefig(SAVE_PATH, dpi=DPI, bbox_inches="tight")
plt.show()
print(f"\nSaved figure to {SAVE_PATH}")